# Notebook 01 - Building Piping Systems with the PipingBuilder DSL

This lesson is the geometry-authoring foundation for the rest of the course. It stays intentionally solver-free: the goal is to build clear, inspectable `TubaModel` geometry before any Code_Aster result workflow.

You will do five things:

1. Build a straight pipe with the cursor-style DSL.
2. Add bends and direction changes.
3. Mix pipe, beam, bar, cable, rectangular, and I-beam sections.
4. Create a branch by starting from an existing node location.
5. Use an interactive geometry-only view to verify local axes before solving.


## 1. Setup

Import Tuba, NumPy, and PyVista, then configure the notebook backend through the shared helper.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pyvista as pv

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model

# Enable interactive notebook rendering
# Defaults to zoomable embedded HTML locally; set TUBA_NOTEBOOK_BACKEND=client or static to override.
from tuba.plotting.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

## 2. The PipingBuilder & Simple Straight Runs

The PipingBuilder is designed as a **fluent interface** that maintains a 3D cursor (position, forward direction, and an up vector). 
You activate the builder using the context manager:
```python
with model.pipe(section=..., material=...) as b:
```
This automatically handles creating nodes, calculating segment lengths, and creating elements on the parent model.

The builder hides most low-level bookkeeping, but the model still stores simple objects:

- `b.start([x, y, z])` starts at a coordinate in meters.
- `b.run(length)` creates the next node in the current forward direction.
- Each created element stores endpoint ids: `n1` for the start node and `n2` for the end node.
- The element vector is `coords(n2) - coords(n1)`, which is what the solver and visualizers use for orientation.

In [ ]:
# Initialize a new model
model = Model("StraightPipe")

# Add basic carbon steel material
model.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)

# Add pipe section profile (DN150 / 6-inch Schedule 40)
model.add_pipe_section("DN150", OD=0.1683, WT=0.0071)

# Construct a 5-meter straight pipe
with model.pipe(section="DN150", material="Steel") as b:
    b.start([0, 0, 0]).run(5.0).end()

print("Nodes:")
for nid, node in model.nodes.items():
    print(f"  Node {nid}: coords={node.coords.tolist()}")

print("\nElements:")
for elem in model.elements:
    start = model.nodes[elem.n1].coords
    end = model.nodes[elem.n2].coords
    vector = end - start
    print(
        f"  Element {elem.id}: type={elem.type}, "
        f"n1={elem.n1}, n2={elem.n2}, vector={vector.tolist()}"
    )


## 3. Bends and Direction Changes

Piping stress analysis relies heavily on elbows/bends to provide flexibility for thermal expansion. 
The PipingBuilder supports calling `.bend(radius, angle, plane)` to insert curved sections and rotate the cursor's forward direction. Supported planes are `"XY"` (horizontal) and `"XZ"` (vertical).

You can also use `.set_direction(vector)` to steer the pipe in arbitrary directions.

In [ ]:
model2 = Model("MultiLegPipe")
model2.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)
model2.add_pipe_section("DN150", OD=0.1683, WT=0.0071)

# Build a 3D routing leg with two 90-degree bends
with model2.pipe(section="DN150", material="Steel") as b:
    b.start([0, 0, 0])
    b.run(3.0)
    b.bend(radius=0.3, angle=90.0, plane="XY")  # turn in horizontal plane
    b.run(2.0)
    b.bend(radius=0.3, angle=90.0, plane="XZ")  # turn upwards
    b.run(1.5)
    b.end()

print("Routing node coordinates:")
for nid, node in model2.nodes.items():
    print(f"  Node {nid}: {node.coords}")

## 4. Modeling Different Section Profiles

Tuba supports coupled structural/piping models. In addition to normal pipes, you can model beams, columns, bar supports, or cables. The sections are defined as:

| Section Class | Description | Section Builder Method |
|---|---|---|
| `PipeSection` | Hollow cylinder with corrosion allowance | `run()` / `run_element(..., 'pipe_straight')` |
| `BarSection` | Solid or hollow cylinder | `bar()` |
| `CableSection` | Tension-only linear elements with pretension | `cable()` |
| `RectangularSection` | Hollow box/rectangular beams | `beam()` |
| `IBeamSection` | standard I-beams loaded from catalog | `beam()` |

Beam-like profiles can also be rotated around the element local X-axis with `twist_angle`. The same twist rotates the rendered profile and the local Y/Z axes, so the local-axis view below is the geometry check before export.

Let's create a model containing all of these section types running parallel to each other, including rotated box and I-beam profiles.

In [ ]:
model3 = Model("MultiSectionDemo")
model3.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)

# Define the section profiles
model3.add_pipe_section("DN100", OD=0.1143, WT=0.006)
model3.add_bar_section("Bar50", OD=0.05, WT=0.0)  # Solid 50mm bar
model3.add_cable_section("Cable10", radius=0.01, pretension=500.0)  # 20mm cable
model3.add_rectangular_section("Box80x40", height_y=0.08, height_z=0.04, thickness_y=0.005, thickness_z=0.005)
model3.add_ibeam_section("IPE100", "IPE100")  # IPE100 standard beam from DB

# 1. Pipe straight run
with model3.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0.0, 0]).run(4.0).end()

# 2. Bar straight run
with model3.pipe(section="Bar50", material="Steel") as b:
    b.start([0, 1.0, 0]).bar(4.0).end()

# 3. Cable straight run
with model3.pipe(section="Cable10", material="Steel") as b:
    b.start([0, 2.0, 0]).cable(4.0).end()

# 4a. Box beam, default local Y/Z axes
with model3.pipe(section="Box80x40", material="Steel") as b:
    b.start([0, 3.0, 0]).beam(4.0).end()

# 4b. Same box beam rotated 45 degrees around local X
with model3.pipe(section="Box80x40", material="Steel") as b:
    b.start([0, 4.0, 0]).beam(4.0, twist_angle=45.0).end()

# 5a. I-beam, default local Y/Z axes
with model3.pipe(section="IPE100", material="Steel") as b:
    b.start([0, 5.0, 0]).beam(4.0).end()

# 5b. Same I-beam rotated 90 degrees around local X
with model3.pipe(section="IPE100", material="Steel") as b:
    b.start([0, 6.0, 0]).beam(4.0, twist_angle=90.0).end()

print(f"Created {len(model3.elements)} parallel elements.")
for elem in model3.elements:
    print(f"  {elem.id:<10s} section={elem.section:<8s} twist_angle={elem.twist_angle:>5.1f} deg")

## 5. Geometry Check: Local Axes and Profile Rotation

This is a geometry validation view, not a solver result. It helps verify beam and rectangular-section local coordinate systems before the model is exported to Code_Aster in later notebooks.

The red arrow is local X along the element. Green and blue are the rotated local Y/Z axes. For the twisted box and I-beam rows, the profile and the green/blue axes rotate together.

In [ ]:
from tuba.plotting.pipeline import build_3d_mesh_from_model
from tuba.plotting.plots import add_local_axes_to_plotter

# True section geometry: each element renders its real profile (hollow pipe,
# box, I-beam, bar, cable) straight from the section definition.
tubes = build_3d_mesh_from_model(model3)

p = pv.Plotter()
p.set_background("#1a1a2e")
p.add_mesh(tubes, color="#cbd5e1", show_edges=True, edge_color="#334155", smooth_shading=True)

# Draw local axes at element midpoints; twist_angle rotates local Y/Z around local X.
add_local_axes_to_plotter(p, model3, scale=0.35)
p.add_axes()
p.show(jupyter_backend=JUPYTER_BACKEND)

## 6. Real Code_Aster End-Force Comparison

The geometry check above proves the profiles and local axes are oriented correctly. This cell solves a small cantilever comparison with Code_Aster: each beam is fixed at the left end and receives the same concentrated downward force at the free end. The different warped shapes come from the section stiffness and `twist_angle`, not from hand-built result values.

In [ ]:
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results

profile_force_model = Model("ProfileEndForceComparison")
profile_force_model.add_material("Steel", E=2.1e11, nu=0.3, rho=7850)
profile_force_model.add_rectangular_section("Box80x40", height_y=0.08, height_z=0.04, thickness_y=0.006, thickness_z=0.006)
profile_force_model.add_ibeam_section("IPE100", "IPE100")

PROFILE_FORCE_SEGMENTS = 12
PROFILE_FORCE_LENGTH = 3.0
PROFILE_FORCE_DEFORM_SCALE = 8.0
profile_specs = [
    (0.0, "Box80x40", 0.0),
    (0.8, "Box80x40", 45.0),
    (1.6, "IPE100", 0.0),
    (2.4, "IPE100", 90.0),
]
cantilever_ends = []
segment_length = PROFILE_FORCE_LENGTH / PROFILE_FORCE_SEGMENTS
for y, section, twist in profile_specs:
    with profile_force_model.pipe(section=section, material="Steel", route=f"{section}-{twist:g}") as b:
        b.start([0.0, y, 0.0], support="anchor")
        for _ in range(PROFILE_FORCE_SEGMENTS):
            b.beam(segment_length, twist_angle=twist)
        b.end()
        cantilever_ends.append((section, twist, b.last_node_id))

end_force = profile_force_model.define_load_case("EndForce", gravity=False)
for _section, _twist, node_id in cantilever_ends:
    end_force.add_nodal_force(node_id, force=[0.0, 0.0, -500.0])

profile_force_model.validate()

CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
# VS Code/Jupyter review defaults to committed real Code_Aster artifacts; set True only after the runtime doctor passes.
RUN_CODE_ASTER = False
PROFILE_FORCE_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "building_profile_end_force"
PROFILE_FORCE_HTML = REPO_ROOT / "notebooks" / "building_profile_end_force.html"

profile_force_run = load_or_run_code_aster_results(
    profile_force_model,
    "EndForce",
    PROFILE_FORCE_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
profile_force_results = profile_force_run.results

print(f"Code_Aster solved: {profile_force_run.ran_solver}")
print(f"Result directory: {profile_force_run.work_dir}")
for section, twist, end_node_id in cantilever_ends:
    disp = profile_force_results.get_displacement(end_node_id)
    print(
        f"section={section:<8s} twist={twist:>5.1f} deg "
        f"free-end DZ={disp[2]: .6e} m"
    )

profile_mesh = build_3d_mesh_from_model(profile_force_model, profile_force_results)
warped = profile_mesh.warp_by_vector("DEPL", factor=PROFILE_FORCE_DEFORM_SCALE) if "DEPL" in profile_mesh.point_data else profile_mesh

p_force = pv.Plotter()
p_force.set_background("#1a1a2e")
p_force.add_mesh(profile_mesh, color="#94a3b8", opacity=0.14, style="wireframe", label="Undeformed")
p_force.add_mesh(
    warped,
    scalars="DEPL_magnitude" if "DEPL_magnitude" in warped.point_data else None,
    cmap="viridis",
    show_edges=True,
    edge_color="#e2e8f0",
    line_width=1.4,
    opacity=0.96,
    smooth_shading=False,
    scalar_bar_args={"title": "Displacement [m]", "color": "white", "fmt": "%.3f"},
    label=f"Deformed x{PROFILE_FORCE_DEFORM_SCALE:g}",
)
p_force.add_axes()
p_force.add_legend(bcolor="#1a1a2e")
p_force.view_vector((1.0, -0.75, 0.42), viewup=(0.0, 0.0, 1.0))
p_force.reset_camera()
p_force.camera.zoom(1.25)
p_force.export_html(str(PROFILE_FORCE_HTML))
print(f"Interactive HTML written: {PROFILE_FORCE_HTML}")
p_force.show(jupyter_backend=JUPYTER_BACKEND)

## 7. Branching Pipes & T-Junctions

When routing complex layouts, you often branch off from an existing pipe run. 
The PipingBuilder is designed to handle this seamlessly: if you start a new pipe run context at coordinates where a node already exists, the builder detects the node and branches off from it.

In [ ]:
model4 = Model("BranchingDemo")
model4.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)
model4.add_pipe_section("DN100", OD=0.1143, WT=0.006)

# 1. Create main straight run (interrupted at the midpoint [3.0, 0, 0] to create a node)
with model4.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0, 0]).run(3.0).run(3.0).end()

# 2. Connect branch starting at [3.0, 0, 0] going vertical (+Z)
with model4.pipe(section="DN100", material="Steel") as b:
    b.start([3.0, 0.0, 0.0]).set_direction([0.0, 0.0, 1.0]).run(2.0).end()

print("Nodes in branching model:")
for nid, node in model4.nodes.items():
    print(f"  Node {nid}: coords={node.coords.tolist()}")

print("\nElements in branching model:")
for elem in model4.elements:
    start = model4.nodes[elem.n1].coords
    end = model4.nodes[elem.n2].coords
    vector = end - start
    print(
        f"  Element {elem.id}: n1={elem.n1}, n2={elem.n2}, "
        f"vector={vector.tolist()}"
    )


## Key Takeaways

- `PipingBuilder` tracks cursor position and direction so you do not hand-calculate 3D routing coordinates.
- `.bend()` creates tangent-continuous direction changes that later export cleanly to Code_Aster.
- `beam()`, `bar()`, and `cable()` can live in the same model as normal pipes.
- `beam(..., twist_angle=...)` rotates beam profiles around local X and changes the local Y/Z axes.
- Local axes visualization is a practical pre-solve geometry check.
- The Code_Aster end-force comparison shows how section stiffness and twist change the warped shape under the same load.

Next: `02_supports_and_loading.ipynb` adds restraints and operating load cases to this geometry model.